In [57]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler,MinMaxScaler

In [83]:
import numpy as np
n_splits=5
seq_length=44
# Load the data
data = np.load("/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/VAEs/vae_generated_data.npy")
feat_dim=17
# Inspect shape
print(data.shape)


(5000, 44, 17)


In [84]:
data=data[:, :, 1:]
y=data[:,:,0]

In [85]:
print(data.shape)

(5000, 44, 16)


In [86]:

data=data.reshape(data.shape[0],-1)
print(data.shape)

(5000, 704)


In [87]:
scaler=MinMaxScaler()
data=scaler.fit_transform(data)

In [88]:
data=data.reshape(data.shape[0],seq_length,16)
print(data.shape)

(5000, 44, 16)


In [89]:
n_splits=5
seq_length=45
df=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data PreProcessing/CLEANED_DATA/train/train_FD001_cleaned.csv',sep=',')
test_df=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/FINAL transformer  PROJECT/data cleaning/test_FD001_cleaned.csv',sep=',')
df_time_warp=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/traditional techniques/time_warping_fd001.csv')
df_mag_warp=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/traditional techniques/magnitude_warping_fd001.csv')
df_scaling=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/traditional techniques/homogeneous_scaling_fd001.csv')
df_window=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/Data Augmentation/traditional techniques/window_permutation_fd001.csv')
rul_true=pd.read_csv('/mnt/c/Users/PC/Downloads/AI/DeepLearning/FINAL transformer  PROJECT/not ready datasets/not ready datasets/RUL_FD001.txt',header=None,names=["RUL"],sep=',')
feature_cols=[col for col in df.columns if not col in ['id','cycle','RUL','setting3']]

In [90]:
columns=['RUL' ,'cycle','setting1', 'setting2', 'setting3','sensor_2', 'sensor_3', 'sensor_4',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12',
       'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20',
       'sensor_21', 'id']
df=df[columns]
df_time_warp=df_time_warp[columns]
df_mag_warp=df_mag_warp[columns]
df_scaling=df_scaling[columns]
df_window=df_window[columns]

In [91]:
df_final = pd.concat(
    [df, df_time_warp, df_mag_warp, df_scaling,df_window],
    ignore_index=True
)

print(df_final.shape)

(103116, 20)


In [92]:
df_final['RUL']=df_final['RUL'].clip(upper=125)

In [93]:
def scaler_process(df,feature_cols):
    x=df[feature_cols]
    y=df['RUL']
    x_scaler=MinMaxScaler()
    x_scaled=x_scaler.fit_transform(x)
    indexes=df[feature_cols]
    scaled_df=pd.DataFrame(
        x_scaled,
        columns=feature_cols,
        index=indexes.index
    )
    scaled_df['cycle']=df['cycle']
    scaled_df['id']=df['id']
    y_clipped = y.clip(upper=125)
    scaled_df['RUL']=y_clipped/125.0
    
    return scaled_df , x_scaler

In [94]:
def scaler_test_process(df,scaler,feature_cols):

    x_scaled=scaler.transform(df[feature_cols])
    indexes=df[feature_cols]
    x_scaled=pd.DataFrame(
        x_scaled,
        columns=feature_cols,
        index=indexes.index
    )
    x_scaled['cycle']=df['cycle']
    x_scaled['id']=df['id']
    return x_scaled

In [95]:
def window_eng_sequences(df, seq_length, feature_cols):
    """
    Transforms a 2D dataframe into 3D array (Samples, Seq_Len, Features)
    using sliding window.
    """
    X, y = [], []

    for engine, engine_data in df.groupby('id'):
        if len(engine_data) < seq_length:
            continue
            
        engine_feat = engine_data[feature_cols].values
        engine_rul = engine_data['RUL'].values

        for i in range(len(engine_data) - seq_length + 1):
            X.append(engine_feat[i : i + seq_length])
            y.append(engine_rul[i + seq_length - 1])
            
    return np.array(X), np.array(y)

In [96]:
def gen_test_windows(df,seq_length, feature_cols,rul_true,RUL_MAX=125):
    X_test = []
    y_test = []

    for engine in df['id'].unique():
        engine_data = df[df['id'] == engine]

        if len(engine_data) >= seq_length:
            X_test.append(engine_data[feature_cols].values[-seq_length:])

            # CMAPSS ground truth is clipped & scaled
            rul = rul_true.loc[engine - 1].values[0]
            rul = min(rul, RUL_MAX) / RUL_MAX
            y_test.append(rul)

    return np.array(X_test), np.array(y_test)    

In [11]:
def evaluate_model(model, X_test, y_test_scaled, RUL_MAX=125):

    y_pred_scaled = model.predict(X_test)
    y_pred_scaled = y_pred_scaled

    y_pred = y_pred_scaled * RUL_MAX

    y_true = y_test_scaled * RUL_MAX
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print("\nModel Performance Metrics (Original RUL scale):")
    print(f"MSE:  {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")

    return y_pred, y_true

In [12]:
def plot_rf_importance(importances, feature_cols, seq_length=50):
    # The importance array is length (50 * 14)
    # We want to aggregate importance by SENSOR (summing across time)
    
    # Reshape back to (Time, Features)
    importance_matrix = importances.reshape(seq_length, len(feature_cols))
    
    # Sum across time axis -> Result shape (14,)
    sensor_importance = np.sum(importance_matrix, axis=0)
    
    # Create DataFrame
    imp_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': sensor_importance
    }).sort_values(by='Importance', ascending=False)
    
    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=imp_df, palette='viridis')
    plt.title("Random Forest: Which Sensors Matter Most?")
    plt.xlabel("Aggregate Importance Score")
    plt.ylabel("Sensor Name")
    plt.grid(True, alpha=0.3)
    plt.show()

# Run this after training
# plot_rf_importance(importances, feature_cols, SEQ_LENGTH_RF)

In [97]:
RUL_MAX = 125
#df=df_final.copy()
df_scaled,scaler=scaler_process(df,feature_cols)
df_scaled.to_csv('df_scaled.csv',index=False)
common_cols=[col for col in df_scaled.columns if not col in ['RUL']]
test_df=test_df[common_cols]

In [98]:
print(df_scaled['RUL'])

0        1.000
1        1.000
2        1.000
3        1.000
4        1.000
         ...  
20626    0.032
20627    0.024
20628    0.016
20629    0.008
20630    0.000
Name: RUL, Length: 20631, dtype: float64


In [99]:
print(len(df_scaled.columns))


19


In [100]:
test_scaled_features=scaler_test_process(test_df,scaler,feature_cols)


In [101]:
print(len(test_scaled_features.columns))

18


In [102]:
test_scaled_df=test_scaled_features.copy()
id_col='id'
if id_col not in (test_scaled_df.columns):
    test_scaled_df['id'] = test_df['id'].values

In [103]:
seq_length=44
X_train,y_train=window_eng_sequences(df_scaled,seq_length,feature_cols)
#X_val,y_val=window_eng_sequences(val_df,seq_length,feature_cols)
X_test,y_test=gen_test_windows(test_scaled_df, seq_length, feature_cols, rul_true, RUL_MAX=125)

In [104]:
print(X_train.shape)
print(X_test.shape)

(16331, 44, 16)
(96, 44, 16)


In [105]:

X_train_aug = np.concatenate([X_train, data], axis=0)
  


In [106]:
# Reduce aug_y to one label per sample, e.g. the last timestep
aug_y_flat = y[:, -1]   # shape (5000,)

# Now both are 1-D
y_train_aug = np.concatenate([y_train, aug_y_flat], axis=0)
print(y_train_aug.shape)  # (21331,)



(21331,)


In [108]:
X_train_flat = X_train_aug.reshape(X_train_aug.shape[0], -1)
print(X_train_flat.shape)
print(y_train_aug.shape)


(21331, 704)
(21331,)


In [109]:
X_test_flat=X_test.reshape(X_test.shape[0],-1)

In [110]:
from sklearn.svm import SVR
SVM_C = 100          # Regularization (Higher = less regularization, fits closer to data)
SVM_EPSILON = 0.1    # Margin of error (Standard for regression)
SVM_KERNEL = 'rbf'
model = SVR(kernel=SVM_KERNEL, C=SVM_C, epsilon=SVM_EPSILON, gamma='scale')


In [111]:
print("Training SVM... (This might take a moment)")
model.fit(X_train_flat, y_train_aug)

Training SVM... (This might take a moment)


,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,100
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [112]:
y_pred, y_test=evaluate_model(model,X_test_flat , y_test, RUL_MAX=125)


Model Performance Metrics (Original RUL scale):
MSE:  230.3806
RMSE: 15.1783
MAE:  12.2388
R²:   0.8555


In [44]:
# 1. Initialize counters OUTSIDE the loop
pessimistic_count = 0
optimistic_count = 0

# 2. Use zip() to pair the lists
for pred, act in zip(y_pred, y_test):
    
    # 3. Compare
    if pred < act:
        pessimistic_count += 1  # Use += to save the value
    else:
        optimistic_count += 1

# 4. Print results
print(f"Pessimistic predictions: {pessimistic_count}")
print(f"Optimistic predictions: {optimistic_count}")

if pessimistic_count > optimistic_count:
    print(" Verdict: Model is Pessimistic (Safer for Maintenance)")
else:
    print(" Verdict: Model is Optimistic (Risky for Maintenance)")

Pessimistic predictions: 44
Optimistic predictions: 52
 Verdict: Model is Optimistic (Risky for Maintenance)


In [45]:
import joblib 

joblib.dump(model, "RF_model_FD001.pkl")
joblib.dump(scaler,"RF_Scaler_FD001.pkl")

['RF_Scaler_FD001.pkl']